In [14]:
import subprocess

In [22]:
netlist = """
Resistor Divider
V1 in 0 DC 5
R1 in out 1k
R2 out 0 1k
.OP
.END
"""

with open("/tmp/test.cir", "w") as f:
    f.write(netlist)

In [23]:
def parse_output(stdout: str) -> dict:
    node_voltages = {}
    in_node_table = False

    for line in stdout.splitlines():
        if "Node" in line and "Voltage" in line:
            in_node_table = True
            continue
        if in_node_table:
            if line.strip()=="" or "----" in line:
                continue
            if "Source" in line or "-----" in line:
                break
            parts = line.split()
            if len(parts) == 2:
                try:
                    node_voltages[parts[0]] = float(parts[1])
                except ValueError:
                    pass
    return node_voltages
        


result = subprocess.run(
    ["ngspice", "-b", "/tmp/test.cir"],
    capture_output=True, text=True
)

In [24]:
nodes = parse_output(result.stdout)

print(nodes)

{'out': 2.5, 'in': 5.0}
